# 🐍 Day 31 · Python Basics — Generators & Streaming (Voice Agent)

> **Goal of this notebook:** A voice agent **streams**: it starts speaking before the full answer is ready. Python does streaming with generators (`yield`). Learn them step by step.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. `return` vs `yield`
2. Using `next()` and `for` with a generator
3. Simulating token streaming
4. Async generators (`async for`)
5. Audio is bytes — chunking bytes
6. 🔗 A mini STT → agent → TTS pipeline

## 1. `return` vs `yield`
- `return` gives back **everything at once** and the function ends.
- `yield` gives back **one piece**, pauses, and continues from there next time. A function with `yield` is a **generator**.

In [ ]:
def all_at_once():
    return ["Hello", "there", "!"]

def one_by_one():
    yield "Hello"
    yield "there"
    yield "!"

print(all_at_once())
print(one_by_one())            # a generator object — nothing has run yet
print(list(one_by_one()))      # list() pulls every value out

## 2. `next()` and `for`

In [ ]:
gen = one_by_one()
print(next(gen))     # Hello
print(next(gen))     # there

for word in one_by_one():      # a for loop calls next() for you
    print("got:", word)

## 3. Simulating token streaming
This is how ChatGPT-style text appears word by word.

In [ ]:
import time

def stream_answer(text):
    for word in text.split():
        time.sleep(0.15)       # pretend the model is thinking
        yield word + " "

for token in stream_answer("Your pizza order is confirmed and will arrive in 30 minutes."):
    print(token, end="", flush=True)     # end="" keeps everything on one line
print()

## 4. Async generators
Real-time servers use `async def` + `yield`, read with `async for`. Now many users can stream at the same time.

In [ ]:
import asyncio

async def astream(text):
    for word in text.split():
        await asyncio.sleep(0.1)
        yield word

async for w in astream("streaming with async for"):
    print(w.upper(), end=" ")
print()

## 5. Audio is bytes
Microphone audio arrives as **bytes** in small chunks (for example every 50 ms).

In [ ]:
audio = bytes(range(256)) * 4             # 1024 pretend audio bytes
print(len(audio), "bytes")

def chunk_bytes(data, size=300):
    for start in range(0, len(data), size):
        yield data[start:start + size]

for i, chunk in enumerate(chunk_bytes(audio)):
    print(f"chunk {i}: {len(chunk)} bytes")

## 6. 🔗 A mini voice pipeline
Each stage is a generator that **reads the previous stage while it is still running**. `pipeline.py` connects STT → agent → TTS in the same way.

In [ ]:
from dataclasses import dataclass

@dataclass
class Event:
    type: str
    data: str

def stt(audio_chunks):                       # speech -> text
    pretend_words = ["one", "large", "pizza", "please"]
    for chunk, word in zip(audio_chunks, pretend_words):
        yield Event("stt", word)

def agent(events):                           # text -> reply tokens
    heard = [e.data for e in events]
    yield Event("agent", f"Ordering: {' '.join(heard)}.")
    yield Event("agent", "Anything else?")

def tts(events):                             # reply -> audio bytes
    for e in events:
        yield Event("tts", f"<{len(e.data.encode())} bytes of audio for '{e.data}'>")

for event in tts(agent(stt(chunk_bytes(audio)))):
    print(event)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Write a generator `countdown(n)` that yields n, n-1, ... 1.
2. Write `stream_letters(word)` that yields one letter at a time and print them with `end="-"`.
3. Change `chunk_bytes` size to 100. How many chunks do you get?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
def countdown(n):
    while n > 0:
        yield n
        n -= 1
print(list(countdown(5)))

def stream_letters(word):
    for ch in word:
        yield ch
for ch in stream_letters("voice"):
    print(ch, end="-")
print()

print(len(list(chunk_bytes(audio, size=100))))   # 11
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Generator | `def gen(): yield x` |
| Next value | `next(gen)` |
| Stream print | `print(token, end='', flush=True)` |
| Async generator | `async def g(): yield x` |
| Async loop | `async for x in g():` |
| Byte chunks | `data[start:start + size]` |

## ➡️ What's next?
You now know the Python used in **Day_31_Project_Voice_Agent**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀